[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/laboratorios/S07/S07_04_roc_pr_desbalance.ipynb)

# S07 · 04 — Curvas ROC y PR, y qué hacer con clases desbalanceadas

**Tiempo estimado:** 60 min · **Sprint 7** · M3 (UD 3.4)

**Contexto TurisData.** Imagina un alojamiento con clientes muy fieles donde **solo el 6 %**
de las reservas se cancela. Un modelo que diga siempre «no cancelará» acertaría el 94 %…
y sería completamente inútil. Cuando el evento que nos importa es **raro** (cancelaciones,
fraudes, averías), la exactitud engaña y hay que usar otras herramientas.

**Al terminar sabrás:**
1. Detectar la «paradoja de la exactitud».
2. Interpretar y calcular las curvas **ROC** (AUC) y **precisión–recall** (AP).
3. Aplicar tres remedios: pesos por clase, submuestreo y ajuste del umbral.

> **Nota honesta:** `reservas_turisdata.csv` tiene un 46 % de cancelaciones. Para este
> laboratorio **simulamos** un escenario raro conservando solo una parte de las canceladas.
> Es un recurso didáctico: en un proyecto real no se manipula la tasa de la población de test.

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.utils import resample
from sklearn.base import clone
from sklearn.metrics import (roc_curve, roc_auc_score, precision_recall_curve, average_precision_score,
                             precision_score, recall_score, accuracy_score)

warnings.filterwarnings("ignore", category=UserWarning)
SEMILLA = 42
sns.set_theme(style="whitegrid")

df = pd.read_csv(dato("reservas_turisdata.csv"), parse_dates=["fecha_reserva", "fecha_llegada"])
df["temporada_alta"] = df["fecha_llegada"].dt.month.isin([1, 2, 3, 7, 8, 12]).astype(int)

# Escenario simulado: conservamos todas las no canceladas y una parte de las canceladas (≈ 6 %)
neg = df[df["cancelada"] == 0]
pos = df[df["cancelada"] == 1].sample(n=int(len(neg) * 0.064), random_state=SEMILLA)
raro = pd.concat([neg, pos]).sample(frac=1, random_state=SEMILLA).reset_index(drop=True)
print(f"Reservas: {len(raro)} · canceladas: {raro['cancelada'].sum()} ({raro['cancelada'].mean():.1%})")

columnas_num = ["antelacion_dias", "noches", "adultos", "ninos", "precio_noche",
                "peticiones_especiales", "cliente_repetidor", "cancelaciones_previas", "temporada_alta"]
columnas_cat = ["tipo_habitacion", "regimen", "canal", "pais_origen", "tarifa", "deposito"]
X, y = raro[columnas_num + columnas_cat], raro["cancelada"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=SEMILLA, stratify=y)

prep = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("esc", StandardScaler())]), columnas_num),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]), columnas_cat),
])

def nuevo_modelo(**kwargs):
    # clone(prep): cada modelo necesita su propia copia del preprocesado (si no, se pisarían entre sí)
    return Pipeline([("prep", clone(prep)), ("clf", LogisticRegression(max_iter=1000, **kwargs))])

## 1. La paradoja de la exactitud

**Ejercicio 1.** Entrena un `DummyClassifier(strategy="most_frequent")` con train. Guarda en
`acc_tonto` su exactitud en test y en `recall_tonto` su recall en test (`recall_score`).
¿Qué te dice?

In [ ]:
# TODO: DummyClassifier(strategy="most_frequent").fit(X_train, y_train); recall_score(y_test, pred)
...
assert acc_tonto > 0.9, "El modelo que dice siempre «no cancela» acierta más del 90 %"
assert recall_tonto == 0, "…pero no detecta ninguna cancelación (recall = 0)"
print(f"Exactitud del modelo inútil: {acc_tonto:.1%} · recall: {recall_tonto:.0%}")

## 2. Curva ROC y AUC

**Idea.** Para **cada umbral posible** medimos:
- **TPR** (tasa de verdaderos positivos = recall): cancelaciones detectadas,
- **FPR** (tasa de falsos positivos): reservas que se mantienen pero marcamos como riesgo.

La **curva ROC** dibuja TPR frente a FPR. Un modelo inútil va por la diagonal; uno bueno se
pega a la esquina superior izquierda. El **AUC** (área bajo la curva) resume: 0,5 = azar; 1 = perfecto.
Interpretación intuitiva: *la probabilidad de que una reserva cancelada reciba una puntuación
más alta que una que no se cancela*.

In [ ]:
base = nuevo_modelo().fit(X_train, y_train)
proba_base = base.predict_proba(X_test)[:, 1]

**Ejercicio 2.** Con `y_test` y `proba_base` calcula `fpr, tpr, _ = roc_curve(...)` y
`auc = roc_auc_score(...)`.

In [ ]:
# TODO: roc_curve(y_test, proba_base) devuelve tres arrays; roc_auc_score(y_test, proba_base)
...
assert len(fpr) == len(tpr) and fpr[0] == 0 and tpr[-1] == 1, "fpr y tpr deben ser la curva ROC completa"
assert 0.6 < auc < 0.95, "El AUC de este modelo debería estar entre 0,6 y 0,95"

plt.figure(figsize=(5.5, 5))
plt.plot(fpr, tpr, color="#2a6f97", lw=2, label=f"Regresión logística (AUC = {auc:.2f})")
plt.plot([0, 1], [0, 1], "--", color="gray", label="Azar (AUC = 0,50)")
plt.title("Curva ROC")
plt.xlabel("Tasa de falsos positivos (falsas alarmas)")
plt.ylabel("Tasa de verdaderos positivos (recall)")
plt.legend(loc="lower right")
plt.show()

## 3. Curva precisión–recall (PR) y *average precision*

**Problema de la ROC con clases raras:** como el FPR se divide entre muchísimos negativos,
la curva puede parecer muy buena aunque **la mayoría de las alarmas sean falsas**. La curva
**PR** mira precisión frente a recall, solo con la clase que importa. La línea de referencia
no es la diagonal sino la **prevalencia** (el % de positivos): un modelo al azar tiene una
precisión igual a la prevalencia. El resumen es la ***average precision*** (AP).

In [ ]:
prec, rec, _ = precision_recall_curve(y_test, proba_base)
ap = average_precision_score(y_test, proba_base)
prevalencia = y_test.mean()
plt.figure(figsize=(5.5, 5))
plt.plot(rec, prec, color="#2a6f97", lw=2, label=f"Regresión logística (AP = {ap:.2f})")
plt.axhline(prevalencia, ls="--", color="gray", label=f"Azar (= prevalencia {prevalencia:.2f})")
plt.title("Curva precisión–recall")
plt.xlabel("Recall (cancelaciones detectadas)")
plt.ylabel("Precisión (alarmas acertadas)")
plt.legend()
plt.show()

**Ejercicio 3.** Guarda en `ap_base` la *average precision* de `proba_base`, y en `veces_mejor_que_azar`
cuántas veces mejor es que la prevalencia (`ap_base / prevalencia`).

In [ ]:
# TODO: average_precision_score(y_test, proba_base); prevalencia ya está calculada
...
assert 0 < ap_base < 1
assert veces_mejor_que_azar > 1.5, "El modelo debería ser claramente mejor que el azar"
print(f"AP = {ap_base:.2f}: el modelo es {veces_mejor_que_azar:.1f} veces mejor que el azar en precisión.")

## 4. Tres remedios para el desbalance

1. **Pesos por clase** (`class_weight="balanced"`): un error en la clase rara cuenta más al entrenar.
2. **Submuestreo** de la clase mayoritaria: entrenar con menos negativos (más equilibrado, pero se tira información).
   *(También existe el sobremuestreo y técnicas como SMOTE, que requieren otra librería y no usamos aquí).*
3. **Ajustar el umbral** de decisión (ya lo hicimos en la práctica anterior): no cambia el modelo, cambia la decisión.

Regla clave: **el remuestreo solo se aplica al conjunto de entrenamiento**, nunca al de test.

**Ejercicio 4.** Crea `con_pesos = nuevo_modelo(class_weight="balanced")`, ajústalo con train y
guarda en `recall_base` y `recall_pesos` el recall en test (umbral 0,5) del modelo base y del
ponderado. Guarda también `precision_pesos`.

In [ ]:
# TODO: nuevo_modelo(class_weight="balanced").fit(...); recall_score(y_test, modelo.predict(X_test))
...
assert recall_pesos > recall_base, "Con pesos por clase el recall debería subir"
assert recall_pesos > 0.5, "El modelo ponderado debería detectar más de la mitad de las cancelaciones"
print(f"Recall: {recall_base:.0%} → {recall_pesos:.0%} · precisión del ponderado: {precision_pesos:.0%}")

**Ejercicio 5 (submuestreo).** Crea `X_sub, y_sub` a partir **solo de train**: todas las positivas y
una muestra aleatoria de negativas del **doble** de tamaño (`resample(..., replace=False,
n_samples=..., random_state=SEMILLA)`). Entrena `con_submuestreo = nuevo_modelo()` con esos datos.

In [ ]:
# TODO: separa positivas y negativas de train, resample() de las negativas y concatena
...
assert y_sub.sum() * 2 == (y_sub == 0).sum(), "Debe haber el doble de negativas que de positivas"
assert len(X_sub) < len(X_train) / 3, "El submuestreo reduce mucho el tamaño del entrenamiento"
assert recall_score(y_test, con_submuestreo.predict(X_test)) > recall_base, "El submuestreo debería subir el recall"

## 5. Ajustar el umbral con la curva PR

**Ejercicio 6.** El gerente pide **detectar al menos el 70 % de las cancelaciones** con la mayor
precisión posible. Con las probabilidades de validación cruzada de `base` sobre train (`proba_cv`),
usa `precision_recall_curve` y guarda en `umbral_70` el **mayor umbral** cuyo recall sea ≥ 0,70.
Después mide en `recall_test_70` y `precision_test_70` cómo funciona ese umbral en test.

In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=SEMILLA)
proba_cv = cross_val_predict(nuevo_modelo(), X_train, y_train, cv=cv, method="predict_proba")[:, 1]
# TODO: precision_recall_curve devuelve (precision, recall, umbrales); recall tiene un elemento más
...
assert 0 < umbral_70 < 0.5, "Para alcanzar recall 0,70 con clases raras el umbral debe ser bajo (< 0,5)"
assert recall_test_70 > 0.5, "En test el recall debería aproximarse al objetivo del 70 %"
print(f"Umbral {umbral_70:.3f}: recall en test {recall_test_70:.0%}, precisión {precision_test_70:.0%}")

## 6. Mini-reto: comparativa de estrategias

Crea `comparativa`, un DataFrame con una fila por estrategia (`"base"`, `"pesos"`, `"submuestreo"`)
y las columnas `AUC`, `AP` (average precision, con probabilidades) y `recall` y `precision`
(con umbral 0,5), todo en test. Redondea a 3 decimales. Guarda en `ganadora_ap` el nombre de la
estrategia con mayor AP.

In [ ]:
# TODO: recorre un dict {nombre: modelo}; predict_proba para AUC/AP y predict para recall/precisión
...
assert list(comparativa.index) == ["base", "pesos", "submuestreo"]
assert {"AUC", "AP", "recall", "precision"} <= set(comparativa.columns)
assert ganadora_ap in comparativa.index
comparativa

Observa: **el AUC y la AP cambian poco entre estrategias** (el modelo "ordena" parecido);
lo que cambia mucho es el **punto de trabajo** (recall vs. precisión con umbral 0,5). Los pesos
y el submuestreo son, en el fondo, formas indirectas de mover el umbral.

## 7. Para reflexionar

1. Con un 6 % de positivos, ¿por qué la ROC puede parecer optimista y la PR no?
2. Si tu objetivo es no perder ninguna cancelación importante, ¿qué métrica vigilas? ¿Y si cada alarma cuesta una llamada de un comercial?
3. ¿Por qué el submuestreo solo se hace con datos de entrenamiento?
4. Con solo ≈ 60 positivas en test, ¿cuánto te fías de un recall del 70 %? ¿Cómo darías una idea de la incertidumbre?

## 8. Resumen: qué has aprendido
- Con clases raras la **exactitud engaña**; mira recall, precisión, AUC y AP.
- **ROC/AUC** mide la capacidad de ordenar; **PR/AP** es más exigente con clases raras.
- Remedios: pesos por clase, submuestreo (solo en train) y, sobre todo, un **umbral** ajustado al negocio.